# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, inspect, and analyze the FAIR² dataset using the [`mlcroissant`](https://mlcroissant.io) library. All manipulations and references use entity `@id`s as specified by the Croissant schema.

### Dataset Source

The dataset source is provided as a Croissant JSON-LD schema:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Croissant Schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview

Review available record sets, fields, and their IDs. All entities are referenced by their `@id`.

Let's enumerate the record sets, and for each, the available fields and their @id.

In [ ]:
# List all record sets in the dataset with their @id and fields
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"- RecordSet: {rs['@id']}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("  Fields:")
        for f in fields:
            # field may be dict or a string @id
            if isinstance(f, dict):
                field_id = f.get('@id', str(f))
            else:
                field_id = f
            print(f"    - Field @id: {field_id}")
    else:
        print("  (No fields defined)")
    print()

## 3. Data Extraction

For each record set, use its `@id` to extract records and load as a DataFrame. You can choose which record set(s) to work with by their `@id`.

In [ ]:
# Collect all the record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records_iter = dataset.records(record_set=record_set_id)
    records = list(records_iter)
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"  => {len(dataframes[record_set_id])} records loaded. Columns: {list(dataframes[record_set_id].columns)}")
    else:
        print("  => No records found for this record set.")

# Pick the first record set with data for further demonstration
data_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        data_record_set_id = rid
        break
if data_record_set_id is not None:
    print(f"\nColumns in record set '{data_record_set_id}':\n{dataframes[data_record_set_id].columns.tolist()}")
    display(dataframes[data_record_set_id].head())
else:
    print("No usable data found in any record set.")

## 4. Exploratory Data Analysis (EDA)

Apply data processing: filter, normalize, or group based on fields referenced by their `@id`. (Adjust `numeric_field_id` and `group_field_id` to match your dataset's columns.)

In [ ]:
# Attempt to pick a numeric field (by inspecting the dataframe)
df = dataframes[data_record_set_id]

# Try to select an int/float column programmatically
numeric_cols = df.select_dtypes(include=['float', 'int']).columns
if len(numeric_cols) == 0:
    print("No numeric columns detected for EDA.")
else:
    numeric_field_id = numeric_cols[0]  # Use the first numeric column
    threshold = df[numeric_field_id].median()
    print(f"Using field '{numeric_field_id}' as numeric field. Median threshold is {threshold}.")
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())
    
    # Normalize the numeric field for these records
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Try grouping by a likely categorical field (object dtype, not the numeric column)
    object_cols = [col for col in df.select_dtypes(include=['object', 'category']).columns if col != numeric_field_id]
    if object_cols:
        group_field_id = object_cols[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of '{numeric_field_id}' by '{group_field_id}':")
        display(grouped_df.head())
    else:
        print("No suitable categorical field for grouping found.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. Here, we visualize the distribution of the numeric and grouping fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(numeric_cols) > 0:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if object_cols:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Boxplot of {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

This notebook demonstrated how to explore and process a Croissant-compliant biomedical dataset using `mlcroissant`. We loaded data by referencing all schema entities by their `@id`, examined record set structure, performed simple filtering, normalization, and grouping on example fields, and visualized their distributions. For deeper analysis, review the [full schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) for available entities and use their `@id` as needed.